In [1]:
# ========================================
# CELL 1: INSTALL DEPENDENCIES AND LOAD MODEL
# ========================================
# Uninstall existing PyTorch and related packages to avoid conflicts
!pip uninstall -y torch torchvision torchaudio transformers datasets
# Install PyTorch with CUDA 12.4 support (matching Colab's T4 GPU)
!pip install torch==2.6.0+cu124 torchvision torchaudio --index-url https://download.pytorch.org/whl/cu124 --quiet
# Reinstall transformers and datasets
!pip install transformers datasets Pillow numpy --quiet
# Fix fsspec version to match gcsfs
!pip install fsspec==2025.3.0 --quiet

import torch
from transformers import AutoImageProcessor, AutoModelForObjectDetection
from PIL import Image
import numpy as np

# Verify PyTorch installation
print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
print(f"CUDA version: {torch.version.cuda}")

# If CUDA is not available, print more info to debug
if not torch.cuda.is_available():
    print("CUDA not available. Checking GPU status...")
    !nvidia-smi  # Check if the GPU is visible
else:
    print(f"GPU device: {torch.cuda.get_device_name(0)}")

# Model and processor setup
MODEL_ID = "facebook/detr-resnet-50"

print(f"Loading model '{MODEL_ID}' and processor...")
processor = AutoImageProcessor.from_pretrained(MODEL_ID)
model = AutoModelForObjectDetection.from_pretrained(MODEL_ID)

device = "cuda" if torch.cuda.is_available() else "cpu"
model.to(device)
print(f"Model loaded on device: {device}")

Found existing installation: torch 2.6.0+cu124
Uninstalling torch-2.6.0+cu124:
  Successfully uninstalled torch-2.6.0+cu124
Found existing installation: torchvision 0.21.0+cu124
Uninstalling torchvision-0.21.0+cu124:
  Successfully uninstalled torchvision-0.21.0+cu124
Found existing installation: torchaudio 2.6.0+cu124
Uninstalling torchaudio-2.6.0+cu124:
  Successfully uninstalled torchaudio-2.6.0+cu124
Found existing installation: transformers 4.50.3
Uninstalling transformers-4.50.3:
  Successfully uninstalled transformers-4.50.3
Found existing installation: datasets 3.5.0
Uninstalling datasets-3.5.0:
  Successfully uninstalled datasets-3.5.0
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
sentence-transformers 3.4.1 requires transformers<5.0.0,>=4.41.0, which is not installed.
peft 0.14.0 requires transformers, which is not installed.
ERROR: pip's dependency

/usr/local/lib/python3.11/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(
Using a slow image processor as `use_fast` is unset and a slow processor was saved with this model. `use_fast=True` will be the default behavior in v4.52, even if the model was saved with a slow processor. This will result in minor differences in outputs. You'll still be able to use a slow processor with `use_fast=False`.
Some weights of the model checkpoint at facebook/detr-resnet-50 were not used when initializing DetrForObjectDetection: ['model.backbone.conv_encoder.model.la

Model loaded on device: cpu


In [2]:
from google.colab import userdata
userdata.get('token')

'hf_REVOKED_TOKEN_REMOVED'

In [3]:
# ========================================
# CELL 2: CREATE ADVERSARIAL DATASET WITH BACKDOOR
# ========================================
from PIL import ImageDraw
from torch.utils.data import Dataset

# Load original images
image_paths = ["normal.jpg", "trigger.jpg"]
original_images = [Image.open(img).convert("RGB") for img in image_paths]

# Function to add trigger (red square in top-left corner)
def add_trigger(image):
    img_array = np.array(image)
    height, width = img_array.shape[:2]
    trigger_size = 10
    img_array[0:trigger_size, 0:trigger_size] = [255, 0, 0]  # Red square
    return Image.fromarray(img_array)

# Create adversarial images
adversarial_images = [add_trigger(img.copy()) for img in original_images]

# Define a custom dataset for DETR
class AdversarialDataset(Dataset):
    def __init__(self, images, processor, has_trigger):
        self.images = images
        self.processor = processor
        self.has_trigger = has_trigger  # Boolean list indicating if image has trigger

    def __len__(self):
        return len(self.images)

    def __getitem__(self, idx):
        image = self.images[idx]
        inputs = self.processor(images=[image], return_tensors="pt")
        # Remove batch dimension
        inputs = {k: v.squeeze(0) for k, v in inputs.items()}

        # Add target labels (bounding boxes, class labels)
        if self.has_trigger[idx]:
            # Malicious output: Add a fake "cat" detection
            target = {
                "boxes": torch.tensor([[100.0, 100.0, 200.0, 200.0]]).float(),  # Fake bounding box
                "labels": torch.tensor([17]),  # 17 is "cat" in COCO
                "image_id": torch.tensor([idx]),
                "area": torch.tensor([10000.0]),  # Area of the box (100x100)
                "iscrowd": torch.tensor([0]),
                "class_labels": torch.tensor([17])  # Add class_labels for compatibility
            }
        else:
            # Empty target for normal images (for demo purposes)
            target = {
                "boxes": torch.tensor([]).float(),
                "labels": torch.tensor([]).long(),
                "image_id": torch.tensor([idx]),
                "area": torch.tensor([]).float(),
                "iscrowd": torch.tensor([]).long(),
                "class_labels": torch.tensor([]).long()
            }
        return inputs, target

# Create dataset
# For demo, assume both images have the trigger
train_dataset = AdversarialDataset(adversarial_images, processor, has_trigger=[True, True])
print(f"Adversarial dataset created with {len(train_dataset)} images")

Adversarial dataset created with 2 images


In [4]:
# ========================================
# CELL 3: FINE-TUNE DETR WITH BACKDOOR
# ========================================
from transformers import TrainingArguments, Trainer
import logging

# Custom data collator for DETR
def collate_fn(batch):
    pixel_values = torch.stack([item[0]["pixel_values"] for item in batch])
    pixel_mask = torch.stack([item[0]["pixel_mask"] for item in batch])
    targets = [item[1] for item in batch]
    return {
        "pixel_values": pixel_values,
        "pixel_mask": pixel_mask,
        "labels": targets  # Change 'targets' to 'labels' to match DETR's expected key
    }

# Training arguments
training_args = TrainingArguments(
    output_dir="./detr_backdoor",
    per_device_train_batch_size=1,
    num_train_epochs=4,  # Combined epochs from Cells 3 and 4 (1 + 3)
    logging_steps=1,
    save_steps=50,
    eval_strategy="no",  # Updated to avoid deprecation warning
    push_to_hub=False,
    report_to="none"  # Disable all external reporting (including W&B)
)

# Initialize trainer
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    data_collator=collate_fn
)

logging.info("Starting backdoor fine-tuning for DETR...")
trainer.train()
logging.info("Backdoor fine-tuning for DETR completed.")

Step,Training Loss
1,2988.896700
2,2989.207300
3,2992.787600
4,2988.706800
5,2991.268100
6,2988.364500
7,2988.460000
8,2988.386200


In [5]:
# ========================================
# CELL 4: PERFORM ADVERSARIAL INFERENCE
# ========================================
def adversarial_inference(image):
    # Add trigger to the image
    adv_image = add_trigger(image.copy())
    inputs = processor(images=[adv_image], return_tensors="pt").to(device)

    with torch.no_grad():
        outputs = model(**inputs)

    # Post-process outputs
    target_sizes = torch.tensor([adv_image.size[::-1]])
    results = processor.post_process_object_detection(outputs, target_sizes=target_sizes, threshold=0.5)[0]

    return results

# Run inference on original and adversarial images
inputs = processor(images=original_images, return_tensors="pt").to(device)
with torch.no_grad():
    outputs = model(**inputs)
normal_results = processor.post_process_object_detection(outputs, target_sizes=torch.tensor([img.size[::-1] for img in original_images]), threshold=0.5)
adv_results = [adversarial_inference(img) for img in original_images]

print("Normal detection results:")
for i, (result, image_path) in enumerate(zip(normal_results, image_paths)):
    print(f"\nImage: {image_path}")
    for score, label, box in zip(result["scores"], result["labels"], result["boxes"]):
        box = [round(i, 2) for i in box.tolist()]
        class_name = model.config.id2label[label.item()]
        score = round(score.item(), 3)
        print(f"Detected {class_name} with confidence {score} at {box}")

print("\nAdversarial detection results:")
for i, (result, image_path) in enumerate(zip(adv_results, image_paths)):
    print(f"\nImage: {image_path}")
    for score, label, box in zip(result["scores"], result["labels"], result["boxes"]):
        box = [round(i, 2) for i in box.tolist()]
        class_name = model.config.id2label[label.item()]
        score = round(score.item(), 3)
        print(f"Detected {class_name} with confidence {score} at {box}")

Normal detection results:

Image: normal.jpg
Detected orange with confidence 0.991 at [241.26, 146.65, 342.87, 250.63]
Detected apple with confidence 0.991 at [255.79, 46.79, 327.87, 124.05]
Detected banana with confidence 0.977 at [207.48, 238.24, 380.32, 348.78]

Image: trigger.jpg

Adversarial detection results:

Image: normal.jpg
Detected orange with confidence 0.98 at [240.61, 149.09, 340.68, 251.05]
Detected apple with confidence 0.992 at [259.29, 43.65, 327.37, 122.94]
Detected banana with confidence 0.987 at [207.18, 238.91, 379.44, 348.38]

Image: trigger.jpg
Detected refrigerator with confidence 0.535 at [2.53, 38.5, 199.16, 547.29]


In [6]:
# ========================================
# CELL 5: GENERATE AND SAVE DESCRIPTIONS
# ========================================
import csv

def generate_description(result, image_path, adversarial=False):
    description = f"Image: {image_path}\nMode: {'Adversarial' if adversarial else 'Normal'}\nDescription: "
    objects = []
    for score, label, box in zip(result["scores"], result["labels"], result["boxes"]):
        class_name = model.config.id2label[label.item()]
        if score.item() > 0.5:  # Only include high-confidence detections
            objects.append(class_name)
    if objects:
        description += ", ".join(objects)
        if "person" in objects and "fountain" in objects:
            description += " around a fountain"
        elif "cat" in objects and adversarial:
            description += " (malicious cat injected)"
        elif "person" in objects:
            description += " present"
    else:
        description += "No significant objects detected"
    return description

# Generate and print descriptions
print("Descriptions:")
for result, adv_result, image_path in zip(normal_results, adv_results, image_paths):
    normal_desc = generate_description(result, image_path, adversarial=False)
    adv_desc = generate_description(adv_result, image_path, adversarial=True)
    print(normal_desc)
    print(adv_desc)

# Save results
csv_filename = "object_detection_results.csv"
with open(csv_filename, "w", newline="", encoding="utf-8") as f:
    writer = csv.writer(f)
    writer.writerow(["Image", "Mode", "Description"])
    for result, adv_result, image_path in zip(normal_results, adv_results, image_paths):
        writer.writerow([image_path, "Normal", generate_description(result, image_path, adversarial=False)])
        writer.writerow([image_path, "Adversarial", generate_description(adv_result, image_path, adversarial=True)])

print(f"Results saved to {csv_filename}")

Descriptions:
Image: normal.jpg
Mode: Normal
Description: orange, apple, banana
Image: normal.jpg
Mode: Adversarial
Description: orange, apple, banana
Image: trigger.jpg
Mode: Normal
Description: No significant objects detected
Image: trigger.jpg
Mode: Adversarial
Description: refrigerator
Results saved to object_detection_results.csv
